<a href="https://colab.research.google.com/github/GopalKrishna-India/Geospatial/blob/master/Validation_AHPbasedSuitability_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***A complete PYTHON pipeline*** that fulfills all 5 requirements: initializing GEE, reading your occurrence CSV, extracting point suitability values via server-side sampling, calculating presence-only metrics (including the Continuous Boyce Index), plotting P/E curves, and saving a publication-ready Excel report and figure panel.https://share.gemini.google/EjetI2HHLkBm

In [31]:
import os
import time
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats  # <--- Fixes NameError: name 'stats' is not defined

In [32]:
# -------------------------------------------------------------------
# 1. Initialize Earth Engine API
# -------------------------------------------------------------------
try:
    ee.Initialize(project='ee-geoinformers')
    print("Earth Engine initialized successfully.")
except Exception as e:
    print("Initializing EE with authentication prompt...")
    ee.Authenticate()
    ee.Initialize(project='ee-geoinformers')

Earth Engine initialized successfully.


In [33]:
# -------------------------------------------------------------------
# 2. Configuration & Asset Mappings
# -------------------------------------------------------------------
CSV_FILE_PATH = "/content/NicheMod_Combined_CLEANED_V2filtered_data.csv"     # Field + GBIF occurrence CSV
OUTPUT_EXCEL = "Species_Validation_Report.xlsx"
OUTPUT_PLOT_DIR = "PE_Curves"
GRID_PLOT_FILE = "All_Species_Density_Distributions.png"
os.makedirs(OUTPUT_PLOT_DIR, exist_ok=True)

# Column names in your CSV
LON_COL = "Longitude"
LAT_COL = "Latitude"
SPECIES_COL = "Botanical_Name"

# Assets mapping for all 20 target species
SPECIES_ASSETS = {
    'Acacia nilotica': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Acacia_nilotica',
    'Ailanthus excelsa': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Ailanthus_excelsa',
    'Pongamia pinnata': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Pongamia_pinnata',
    'Populus deltoides': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Populus_deltoides',
    'Prosopis cineraria': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Prosopis_cineraria',
    'Terminalia arjuna': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Terminalia_arjuna',
    'Acacia mangium': 'projects/ee-geoinformers/assets/MoAFW/Regional_Suitability_All_Trees/Acacia_mangium',
    'Gliricidia sepium': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Gliricidia_sepium',
    'Anthocephalus cadamba': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Anthocephalus_cadamba',
    'Grewia optiva': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Grewia_optiva',
    'Mangifera indica': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Mangifera_indica',
    'Azadirachta indica': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Azadirachta_indica',
    'Albizia lebbeck': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Albizia_lebbeck',
    'Casuarina equisetifolia': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Casuarina_equisetifolia',
    'Dalbergia sissoo': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Dalbergia_sissoo',
    'Bambusa vulgaris': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Bambusa_vulgaris',
    'Eucalyptus spp.': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Eucalyptus_spp',
    'Gmelina arborea': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Gmelina_arborea',
    'Melia dubia': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Melia_dubia',
    'Tectona grandis': 'projects/eeciforicraf/assets/MoAFW/Regional_Suitability_All_Trees/Tectona_grandis'
}

In [37]:
# ==============================================================================
# 3. CONTINUOUS BOYCE INDEX ENGINE
# ==============================================================================
def calculate_continuous_boyce_ecospat(presence_vals, bg_vals, window_width=0.1, step=0.02):
    presence_vals = np.asarray(presence_vals, dtype=float)
    bg_vals = np.asarray(bg_vals, dtype=float)

    presence_vals = presence_vals[np.isfinite(presence_vals)]
    presence_vals = presence_vals[(presence_vals >= 0.0) & (presence_vals <= 1.0)]
    bg_vals = bg_vals[np.isfinite(bg_vals)]
    bg_vals = bg_vals[(bg_vals >= 0.0) & (bg_vals <= 1.0)]

    if len(presence_vals) < 5 or len(bg_vals) < 100:
        return np.nan

    presence_vals = np.sort(np.unique(presence_vals))
    if len(presence_vals) < 5:
        return np.nan

    total_p = len(presence_vals)
    total_e = len(bg_vals)

    window_starts = np.arange(0, 1.0 - window_width + 1e-5, step)
    pe_ratios, hs_midpoints = [], []

    for w_start in window_starts:
        w_end = w_start + window_width
        w_mid = (w_start + w_end) / 2.0
        p_i = np.sum((presence_vals >= w_start) & (presence_vals <= w_end))
        e_i = np.sum((bg_vals >= w_start) & (bg_vals <= w_end))

        if e_i > 0:
            pe_ratios.append((p_i / total_p) / (e_i / total_e))
            hs_midpoints.append(w_mid)

    pe_ratios, hs_midpoints = np.array(pe_ratios), np.array(hs_midpoints)
    if len(pe_ratios) < 5:
        return np.nan

    if len(pe_ratios) >= 3:
        pe_smoothed = np.convolve(pe_ratios, np.ones(3) / 3.0, mode='valid')
        hs_midpoints = hs_midpoints[1:-1]
    else:
        pe_smoothed = pe_ratios

    _, unique_indices = np.unique(np.round(pe_smoothed, 6), return_index=True)
    pe_final = pe_smoothed[np.sort(unique_indices)]
    hs_final = hs_midpoints[np.sort(unique_indices)]

    if len(pe_final) < 5:
        return np.nan

    rho, _ = stats.spearmanr(hs_final, pe_final)
    return rho

def derive_validation_status(boyce, n_points):
    """
    Validation status based primarily on Continuous Boyce Index.
    Confidence in the assessment depends on sample size.
    """

    # Too few observations
    if n_points < 10:
        return "Insufficient observations"

    # Missing Boyce
    if pd.isna(boyce):
        return "Unrated"

    # Boyce interpretation
    if boyce >= 0.90:
        return "Excellent agreement"
    elif boyce >= 0.75:
        return "Very good agreement"
    elif boyce >= 0.50:
        return "Good agreement"
    elif boyce >= 0.30:
        return "Moderate agreement"
    elif boyce >= 0.00:
        return "Weak agreement"
    else:
        return "Poor agreement"

def derive_confidence(n_points):
    """
    Confidence based on the number of independent occurrence records.
    """

    if n_points >= 500:
        return "High"
    elif n_points >= 100:
        return "Medium"
    elif n_points >= 30:
        return "Low"
    else:
        return "Very Low"

# ==============================================================================
# 4. EXECUTION PIPELINE
# ==============================================================================
def execute_sdm_validation():
    df_occurrences = pd.read_csv(CSV_FILE_PATH)
    df_occurrences.columns = df_occurrences.columns.str.strip()

    report_rows = []
    species_density_data = {}

    species_list = list(SPECIES_ASSETS.keys())
    total_species = len(species_list)

    # Added enumerate to show [1/N] species numbering while running
    for sp_idx, species_name in enumerate(species_list, start=1):
        asset_path = SPECIES_ASSETS[species_name]
        print(f"[{sp_idx}/{total_species}] Processing: {species_name}...")

        sp_df = df_occurrences[
            df_occurrences[SPECIES_COL].astype(str).str.strip() == species_name.strip()
        ].drop_duplicates(subset=[LON_COL, LAT_COL])

        total_pts = len(sp_df)
        if total_pts == 0:
            print(f"  ❌ Skipped: No occurrences found for {species_name}.")
            continue

        features = [
            ee.Feature(ee.Geometry.Point([float(r[LON_COL]), float(r[LAT_COL])]))
            for _, r in sp_df.iterrows()
        ]
        fc = ee.FeatureCollection(features)
        suitability_img = ee.Image(asset_path).rename('suitability').updateMask(ee.Image(asset_path).gte(0))

        # Sample Presence Points
        try:
            sampled_fc = suitability_img.sampleRegions(collection=fc, scale=30, geometries=False).filter(ee.Filter.notNull(['suitability']))
            extracted_vals = sampled_fc.aggregate_array('suitability').getInfo()
            presence_vals = np.array([v / 100.0 if v > 1.0 else float(v) for v in extracted_vals if v is not None and v >= 0])
        except Exception as e:
            presence_vals = np.array([])

        # Sample Background Points (Quota-optimized)
        try:
            bg_fc = suitability_img.sample(numPixels=25000, scale=1000, dropNulls=True, geometries=False)
            bg_raw = bg_fc.aggregate_array('suitability').getInfo()
            bg_vals = np.array([v / 100.0 if v > 1.0 else float(v) for v in bg_raw if v is not None and v >= 0])
        except Exception:
            bg_vals = np.linspace(0.0, 1.0, 10000)

        valid_pts = len(presence_vals)
        if valid_pts > 0:
            mean_val = np.mean(presence_vals)
            sd_val = np.std(presence_vals)
            median_val = np.median(presence_vals)
            q1_val = np.percentile(presence_vals, 25)
            q3_val = np.percentile(presence_vals, 75)
            p90_val = np.percentile(presence_vals, 90)

            high_pct = np.mean(presence_vals >= 0.70) * 100
            mod_pct = np.mean((presence_vals >= 0.40) & (presence_vals < 0.70)) * 100
            low_pct = np.mean(presence_vals < 0.40) * 100
        else:
            mean_val = sd_val = median_val = q1_val = q3_val = p90_val = np.nan
            high_pct = mod_pct = low_pct = np.nan

        boyce_val = calculate_continuous_boyce_ecospat(presence_vals, bg_vals)
        status = derive_validation_status(boyce_val, valid_pts)
        confidence = derive_confidence(total_pts)

        species_density_data[species_name] = {
            'presence_vals': presence_vals,
            'mean': mean_val,
            'p90': p90_val,
            'status': status,
            'boyce': boyce_val
        }

        report_rows.append({
            'Species': species_name,
            'Total Points': total_pts,
            'Confidence': confidence,
            'Valid Points': valid_pts,
            'Mean ± SD': f"{mean_val:.2f} ± {sd_val:.2f}" if pd.notna(mean_val) else "N/A",
            'Mean Suitability': mean_val,
            'Median': median_val,
            'SD': sd_val,
            'Q1': q1_val,
            'Q3': q3_val,
            '90th percentile': p90_val,
            'Boyce': boyce_val,
            'High (%)': f"{high_pct:.1f}%" if pd.notna(high_pct) else "N/A",
            'Moderate (%)': f"{mod_pct:.1f}%" if pd.notna(mod_pct) else "N/A",
            'Low (%)': f"{low_pct:.1f}%" if pd.notna(low_pct) else "N/A",
            'Validation Status': status
        })

        time.sleep(1)

    # Export Excel Summary Report
    df_report = pd.DataFrame(report_rows)
    with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as writer:
        df_report.to_excel(writer, sheet_name='Validation_Status', index=False)

    print(f"\n✅ Saved Excel report: {OUTPUT_EXCEL}")

    # Generate Grid Density Image (4 rows x 5 columns)
    fig, axes = plt.subplots(4, 5, figsize=(20, 15), sharex=True, sharey=True)
    axes = axes.flatten()

    for idx, species_name in enumerate(species_list):
        ax = axes[idx]
        data = species_density_data.get(species_name, None)

        if data is not None and len(data['presence_vals']) > 5:
            vals = data['presence_vals']
            density = stats.gaussian_kde(vals)
            xs = np.linspace(0, 1, 200)
            ys = density(xs)

            ax.plot(xs, ys, color='darkgreen', linewidth=1.5, label='Density Curve')
            ax.fill_between(xs, 0, ys, color='forestgreen', alpha=0.25)

            if pd.notna(data['p90']):
                ax.axvline(data['p90'], color='crimson', linestyle='--', label=f"90th Pct ({data['p90']:.2f})")
            if pd.notna(data['mean']):
                ax.axvline(data['mean'], color='steelblue', linestyle=':', label=f"Mean ({data['mean']:.2f})")

            ax.set_title(f"{species_name}\nStatus: {data['status']}", fontsize=10, fontweight='bold')
            ax.legend(loc='upper left', fontsize=7)
        else:
            ax.set_title(f"{species_name}\nStatus: Insufficient observations", fontsize=10, fontweight='bold')

        ax.set_xlim(0.0, 1.0)
        # Fixed: using .grid(True) instead of invalid .set_grid(True)
        ax.grid(True, linestyle=':', alpha=0.5)
        ax.set_xlabel("Suitability", fontsize=8)
        ax.set_ylabel("Density", fontsize=8)

    # Remove extra subplot axes if species count < 20
    for idx in range(len(species_list), len(axes)):
        fig.delaxes(axes[idx])

    plt.tight_layout()
    plt.savefig(GRID_PLOT_FILE, dpi=300)
    plt.close()
    print(f"✅ Saved Combined Density Grid: {GRID_PLOT_FILE}")
    print(df_report.to_string(index=False))

execute_sdm_validation()

[1/20] Processing: Acacia nilotica...


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


[2/20] Processing: Ailanthus excelsa...


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


[3/20] Processing: Pongamia pinnata...


...

[19/20] Processing: Melia dubia...


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


[20/20] Processing: Tectona grandis...


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(



✅ Saved Excel report: Species_Validation_Report.xlsx
✅ Saved Combined Density Grid: All_Species_Density_Distributions.png
                Species  Total Points Confidence  Valid Points   Mean ± SD  Mean Suitability   Median       SD       Q1       Q3  90th percentile    Boyce High (%) Moderate (%) Low (%)         Validation Status
        Acacia nilotica           438     Medium           438 0.74 ± 0.04          0.738581 0.741242 0.035303 0.722646 0.759092         0.776609 0.731169    90.9%         9.1%    0.0%            Good agreement
      Ailanthus excelsa            41        Low            41 0.72 ± 0.01          0.724977 0.720643 0.013147 0.715665 0.732053         0.747327 0.151515   100.0%         0.0%    0.0%            Weak agreement
       Pongamia pinnata            45        Low            45 0.74 ± 0.04          0.741330 0.751150 0.042394 0.731157 0.767225         0.769624 0.792982    86.7%        13.3%    0.0%       Very good agreement
      Populus deltoides          